### Data Reading

In [0]:
bronze_path = "abfss://automotive-sales@automotivesalessa.dfs.core.windows.net/bronze"
silver_path = "abfss://automotive-sales@automotivesalessa.dfs.core.windows.net/silver"

In [0]:
df_data = (
    spark.read.format("parquet")
    .option("header", True)
    .option("inferSchema", True)
    .load(f"{bronze_path}/raw-data")
)

### Data Transformation

In [0]:
from pyspark.sql.functions import *

In [0]:
df_data_tf = (
    df_data
    .withColumn("Model_Category", split(col("Model_ID"),"-")[0])
)

In [0]:
df_data_tf = (
    df_data_tf
    .withColumn("Per_Unit_Revenue", col("Revenue")/col("Units_Sold"))
)

AD-HOC

In [0]:
df_data_tf_agg = (
    df_data_tf
    .groupBy("Year", "BranchName").agg(
        sum("Units_Sold").alias("Total_Units_Sold")
    )
    .sort("Year","Total_Units_Sold", ascending=[True, False])
)

Data Writing

In [0]:
(
    df_data_tf.write
    .format("parquet")
    .mode("overwrite")
    .option("path", f"{silver_path}/car_sales")
    .save()
)